## Cell 1 — Imports & Device Setup

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss
from transformers import DebertaV2Tokenizer, DebertaV2Model
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.utils.class_weight import compute_class_weight  # FIX 2: weighted loss
from transformers import get_linear_schedule_with_warmup
import numpy as np

print(f"NumPy  : {np.__version__}")
print(f"Pandas : {pd.__version__}")
print(f"PyTorch: {torch.__version__}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")


NumPy  : 1.26.4
Pandas : 2.1.4
PyTorch: 2.8.0+cu128
Device : cuda


## Cell 2 — Data Loading (Full Dataset)

> **FIX 1**: Removed `df.sample(frac=0.116)`. The full dataset is now used. Keeping only 11.6% of the data (≈768 rows) was starving the model of signal.


In [ ]:
df = pd.read_csv("allsides.csv")
df = df.drop(["roundup", "Stance"], axis=1)
df = df.dropna()

# FIX 1: Use full dataset — removed df.sample(frac=0.116) which was
# discarding ~88% of training data (768 → ~6600+ samples)
df_sample = df.reset_index(drop=True)

print(f"Full Dataset Size : {len(df_sample)}")
df_sample.sample()


Sample Size : 1064


,News_ID,Title,News_Body,Label,issue,topic
803,1446,"U.S. to Put Tariffs on Chinese Goods, Drawing ...",The Trump administration said on Friday that i...,liberal,Trump Announces Tariffs on 50 Billion in Chine...,Trade


## Cell 3 — Label Encoding

In [3]:
label_map = {
    "liberal"      : 0,
    "center"       : 1,
    "conservative" : 2
}

print(f"Unique Labels in Dataset : {df_sample['Label'].unique()}")

# Drop rows with labels not in the map
df_sample = df_sample[df_sample["Label"].isin(label_map.keys())].reset_index(drop=True)

labels = df_sample["Label"].map(label_map).values

print(f"\nAfter Cleaning:")
print(f"  Remaining Rows : {len(df_sample)}")
print(f"  Labels Shape   : {labels.shape}")
print(f"  Unique Values  : {sorted(set(labels))}")
print(f"  Label Counts   : Left={sum(labels==0)}, Center={sum(labels==1)}, Right={sum(labels==2)}")

Unique Labels in Dataset : ['conservative' 'center' 'liberal']

After Cleaning:
  Remaining Rows : 1064
  Labels Shape   : (1064,)
  Unique Values  : [0, 1, 2]
  Label Counts   : Left=323, Center=368, Right=373


## Cell 4 — Token Extractor (REDESIGN: token sequences, not CLS pooling)

> **REDESIGN**: `EmbeddingExtractor` → `TokenExtractor`.
> Extracts `last_hidden_state` at fixed per-field sequence lengths instead of a
> single CLS vector. This gives the attention mechanisms actual sequences to attend over,
> fixing the gradient vanishing root cause identified in §5 of the report.


In [4]:
class TokenExtractor:
    """
    REDESIGN: Extracts token-level hidden states from DeBERTa
    instead of CLS-pooled embeddings.

    Returns last_hidden_state tensors:
      issue   → [B, Li, 768]  (max_length=16)
      topic   → [B, Lt, 768]  (max_length=16)
      title   → [B, Lh, 768]  (max_length=32)
      body    → [B, Lb, 768]  (max_length=128)

    Why this fixes the attention collapse:
    - Old pipeline: unsqueeze(1) → [B, 1, 768] → attention over 1 token = trivial
    - New pipeline: [B, N, 768] → attention over N tokens = meaningful sequences
    """
    # Max token lengths per field (spec §7)
    MAX_LENGTHS = {
        "issue"    : 16,
        "topic"    : 16,
        "Title"    : 32,
        "News_Body": 128,
    }

    def __init__(self, model_name="microsoft/deberta-v3-base", batch_size=16,
                 unfreeze_top_layers=4):
        self.device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.batch_size = batch_size
        print(f"Loading DeBERTa on {self.device}...")
        self.tokenizer = DebertaV2Tokenizer.from_pretrained(model_name)
        self.model     = DebertaV2Model.from_pretrained(model_name).to(self.device)

        # Freeze all layers first
        for param in self.model.parameters():
            param.requires_grad = False

        # Unfreeze top-N transformer layers (differential LR in training)
        encoder_layers = self.model.encoder.layer
        num_layers     = len(encoder_layers)
        for layer in encoder_layers[num_layers - unfreeze_top_layers:]:
            for param in layer.parameters():
                param.requires_grad = True
        if hasattr(self.model.encoder, "rel_embeddings"):
            for param in self.model.encoder.rel_embeddings.parameters():
                param.requires_grad = True

        frozen    = sum(p.numel() for p in self.model.parameters() if not p.requires_grad)
        trainable = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        print(f"DeBERTa — Frozen: {frozen:,}  |  Trainable top-{unfreeze_top_layers} layers: {trainable:,}")
        print("TokenExtractor ready.")

    def _extract_tokens(self, texts, max_length):
        """Return last_hidden_state as a numpy array [N, max_length, 768]."""
        all_hidden = []
        self.model.eval()
        with torch.no_grad():
            for i in range(0, len(texts), self.batch_size):
                batch  = texts[i : i + self.batch_size]
                inputs = self.tokenizer(
                    batch,
                    padding    = "max_length",
                    truncation = True,
                    max_length = max_length,
                    return_tensors = "pt"
                ).to(self.device)
                outputs    = self.model(**inputs)
                hidden     = outputs.last_hidden_state  # [b, max_length, 768]
                all_hidden.append(hidden.cpu().numpy())
        return np.vstack(all_hidden)  # [N, max_length, 768]

    def extract(self, df, columns=["issue", "topic", "Title", "News_Body"]):
        results = {}
        for col in columns:
            max_len = self.MAX_LENGTHS[col]
            print(f"  Extracting {col:10s} (max_len={max_len})...")
            texts        = df[col].fillna("").astype(str).tolist()
            results[col] = self._extract_tokens(texts, max_len)
            print(f"  Done — Shape: {results[col].shape}")
        return results

    def get_trainable_params(self):
        """Return unfrozen DeBERTa params for the differential-LR optimizer group."""
        return [p for p in self.model.parameters() if p.requires_grad]


## Cell 5 — Extract & Save Token Sequences


In [5]:
extractor = TokenExtractor(batch_size=16)
token_seqs = extractor.extract(df_sample)

issue_tokens = token_seqs["issue"]      # [N, 16,  768]
topic_tokens = token_seqs["topic"]      # [N, 16,  768]
title_tokens = token_seqs["Title"]      # [N, 32,  768]
body_tokens  = token_seqs["News_Body"]  # [N, 128, 768]

print("\nAll Token Sequence Shapes:")
print(f"  Issue  : {issue_tokens.shape}")
print(f"  Topic  : {topic_tokens.shape}")
print(f"  Title  : {title_tokens.shape}")
print(f"  Body   : {body_tokens.shape}")

np.save("issue_tokens.npy", issue_tokens)
np.save("topic_tokens.npy", topic_tokens)
np.save("title_tokens.npy", title_tokens)
np.save("body_tokens.npy",  body_tokens)
np.save("labels.npy",       labels)
print("\nAll token sequences saved to disk.")


Loading DeBERTa on cuda...


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model Ready.
  Extracting: issue...
  Done — Shape: (1064, 768)
  Extracting: topic...
  Done — Shape: (1064, 768)
  Extracting: Title...
  Done — Shape: (1064, 768)
  Extracting: News_Body...
  Done — Shape: (1064, 768)

All Embedding Shapes:
  Issue  : (1064, 768)
  Topic  : (1064, 768)
  Title  : (1064, 768)
  Body   : (1064, 768)

All embeddings saved to disk.


## Cell 6 — Dataset & DataLoaders (3-D token sequences)

> **REDESIGN**: `AllSidesDataset` now stores and returns `[L, 768]` token tensors per sample.
> DataLoader stacks these to `[B, L, 768]` — the shape cross-attention requires.


In [6]:
class AllSidesDataset(Dataset):
    """
    REDESIGN: Stores token sequences [L, 768] per sample instead of
    flat [768] pooled vectors. DataLoader will collate to [B, L, 768].
    """
    def __init__(self, issue, topic, title, body, labels):
        self.issue  = torch.FloatTensor(issue)   # [N, 16,  768]
        self.topic  = torch.FloatTensor(topic)   # [N, 16,  768]
        self.title  = torch.FloatTensor(title)   # [N, 32,  768]
        self.body   = torch.FloatTensor(body)    # [N, 128, 768]
        self.labels = torch.LongTensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "issue"  : self.issue[idx],   # [16,  768]
            "topic"  : self.topic[idx],   # [16,  768]
            "title"  : self.title[idx],   # [32,  768]
            "body"   : self.body[idx],    # [128, 768]
            "label"  : self.labels[idx]
        }


# ── Load token sequences from disk ──────────────────────────────────────────
issue_tokens = np.load("issue_tokens.npy")
topic_tokens = np.load("topic_tokens.npy")
title_tokens = np.load("title_tokens.npy")
body_tokens  = np.load("body_tokens.npy")
labels       = np.load("labels.npy")

indices = np.arange(len(labels))

train_val_idx, test_idx = train_test_split(
    indices, test_size=0.15, random_state=42, stratify=labels
)
train_idx, val_idx = train_test_split(
    train_val_idx, test_size=0.15, random_state=42, stratify=labels[train_val_idx]
)

full_dataset  = AllSidesDataset(issue_tokens, topic_tokens, title_tokens, body_tokens, labels)
train_dataset = Subset(full_dataset, train_idx)
val_dataset   = Subset(full_dataset, val_idx)
test_dataset  = Subset(full_dataset, test_idx)

BATCH_SIZE   = 16
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Train : {len(train_dataset)} samples  ({len(train_loader)} batches)")
print(f"Val   : {len(val_dataset)} samples  ({len(val_loader)} batches)")
print(f"Test  : {len(test_dataset)} samples  ({len(test_loader)} batches)")
print(f"\nSample batch issue shape  : [B={BATCH_SIZE}, Li=16,  D=768]")
print(f"Sample batch body shape   : [B={BATCH_SIZE}, Lb=128, D=768]")


Train Samples : 768
Val Samples   : 136
Test Samples  : 160
Train Batches : 48
Val Batches   : 9
Test Batches  : 10


## Cell 7 — Issue-Topic Context Encoder (REDESIGN: token sequences)

> **REDESIGN**: Input changed from two CLS vectors `[B, 768]` → token sequences
> `[B, Li, 768]` and `[B, Lt, 768]`. They are concatenated to `[B, Li+Lt, 768]`
> before encoding. Output is now a proper contextual sequence, not a 2-token stub.


In [7]:
class IssueTopicContextEncoder(nn.Module):
    """
    REDESIGN: Accepts token sequences instead of single pooled vectors.

    Input:
        issue_tokens : [B, Li, 768]
        topic_tokens : [B, Lt, 768]

    Processing:
        Concatenate along sequence dim → [B, Li+Lt, 768]
        Pass through lightweight TransformerEncoder (2L, 4H, ff=1536)

    Output:
        issue_topic_context : [B, Li+Lt, 768]

    This produces a proper contextual sequence for cross-attention queries,
    replacing the previous [B, 2, 768] two-token representation.
    """
    def __init__(
        self,
        d_model    = 768,
        num_heads  = 4,
        ff_dim     = 1536,
        num_layers = 2,
        dropout    = 0.10
    ):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model         = d_model,
            nhead           = num_heads,
            dim_feedforward = ff_dim,
            dropout         = dropout,
            activation      = "gelu",
            batch_first     = True,
            norm_first      = True   # Pre-LN for stability
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

    def forward(self, issue_tokens, topic_tokens):
        # issue_tokens : [B, Li, 768]
        # topic_tokens : [B, Lt, 768]
        x       = torch.cat([issue_tokens, topic_tokens], dim=1)  # [B, Li+Lt, 768]
        context = self.encoder(x)                                  # [B, Li+Lt, 768]
        return context


## Cell 8 — Cross-Attention Encoder (REDESIGN: multi-token sequences)

> **REDESIGN**: Query is now `[B, Li+Lt, 768]` (full context sequence) and
> Key/Value is `[B, Lh/Lb, 768]` (full headline/body token sequence).
> Attention is no longer trivially over a single token — gradients now flow.
> Output `[B, Li+Lt, 768]` is passed to `AttentionPooling` (new, §11 of spec).


In [8]:
class CrossAttentionEncoderBlock(nn.Module):
    """
    REDESIGN: Cross-attention now operates on real token sequences.

    Query     : issue_topic_context  [B, Li+Lt, 768]
    Key/Value : headline or body     [B, Lh/Lb, 768]

    Both query and key/value are multi-token sequences, so attention
    can learn meaningful cross-sequence relationships (§9, §10 of spec).
    """
    def __init__(self, d_model=768, num_heads=8, ff_dim=3072, dropout=0.10):
        super().__init__()
        self.cross_attention = nn.MultiheadAttention(
            embed_dim   = d_model,
            num_heads   = num_heads,
            dropout     = dropout,
            batch_first = True
        )
        self.norm1   = nn.LayerNorm(d_model)
        self.norm2   = nn.LayerNorm(d_model)
        self.ffn     = nn.Sequential(
            nn.Linear(d_model, ff_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, d_model)
        )
        self.dropout = nn.Dropout(dropout)
        # Interpretability: retain last attention weights
        self.last_attn_weights = None

    def forward(self, query, key_value):
        # query     : [B, Li+Lt, 768]
        # key_value : [B, Lh or Lb, 768]
        attn_out, attn_weights = self.cross_attention(
            query=query, key=key_value, value=key_value
        )
        self.last_attn_weights = attn_weights.detach()

        x      = self.norm1(query + self.dropout(attn_out))
        output = self.norm2(x + self.dropout(self.ffn(x)))
        return output, attn_weights


class CrossAttentionEncoder(nn.Module):
    """
    Stack of CrossAttentionEncoderBlocks.
    Output shape: [B, Li+Lt, 768] — pooled downstream by AttentionPooling.
    """
    def __init__(self, num_layers=4, d_model=768, num_heads=8, ff_dim=3072, dropout=0.10):
        super().__init__()
        self.layers = nn.ModuleList([
            CrossAttentionEncoderBlock(d_model=d_model, num_heads=num_heads,
                                       ff_dim=ff_dim, dropout=dropout)
            for _ in range(num_layers)
        ])

    def forward(self, query, key_value):
        # query     : [B, Li+Lt, 768]
        # key_value : [B, Lh or Lb, 768]
        x = query
        for layer in self.layers:
            x, _ = layer(x, key_value)
        return x  # [B, Li+Lt, 768]  — NOT pooled here; pooling is in AttentionPooling

    def get_last_attn_weights(self):
        return self.layers[-1].last_attn_weights


## Cell 9 — Attention Pooling (NEW: replaces mean pooling, spec §11)

> **NEW**: Learnable attention pooling collapses `[B, N, 768]` → `[B, 768]`.
> A `Linear(768→1)` scores each token position; `Softmax` normalises; output
> is a weighted sum. Applied independently to `headline_context` and `body_context`.
> This lets the model learn *which contextual positions are most informative* for
> bias prediction, instead of uniformly averaging all positions.


In [ ]:
class AttentionPooling(nn.Module):
    """
    Learnable attention pooling: [B, N, 768] → [B, 768]

    score_i = Linear(768 → 1)
    α       = Softmax(score)           # [B, N, 1]
    out     = Σ α_i · h_i              # [B, 768]

    Replaces mean pooling in CrossAttentionEncoder output.
    Spec §11: allows model to weight contextually important positions.
    """
    def __init__(self, d_model=768):
        super().__init__()
        self.score = nn.Linear(d_model, 1)

    def forward(self, x):
        # x : [B, N, 768]
        scores  = self.score(x)              # [B, N, 1]
        weights = torch.softmax(scores, dim=1)  # [B, N, 1]
        pooled  = (weights * x).sum(dim=1)   # [B, 768]
        return pooled


## Cell 10 — Gated Fusion (+ Interpretability Hooks)

> Gate statistics (`mean`, `std`) stored after every forward pass.
> `gate_std < 0.05` → collapse warning during training.


In [ ]:
class GatedFusion(nn.Module):
    """
    Adaptive gated fusion: headline_embedding + body_embedding → fused [B, 768]

    Gate  g  = σ( W2 · GELU( W1 · [H; B] ) )
    Fused F  = g × H + (1 − g) × B
    """
    def __init__(self, hidden_dim=768, dropout=0.10):
        super().__init__()
        self.gate_network = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Sigmoid()
        )
        self.last_gate_mean = None
        self.last_gate_std  = None

    def forward(self, headline_emb, body_emb):
        combined = torch.cat([headline_emb, body_emb], dim=-1)  # [B, 1536]
        gate     = self.gate_network(combined)                   # [B, 768]
        with torch.no_grad():
            self.last_gate_mean = gate.mean().item()
            self.last_gate_std  = gate.std().item()
        fused = gate * headline_emb + (1.0 - gate) * body_emb
        return fused


## Cell 11 — Bias Classifier Head


In [ ]:
class BiasClassifier(nn.Module):
    """MLP classifier: 768 → 512 → 256 → 3  (GELU, dropout=0.30/0.20)"""
    def __init__(self, input_dim=768, num_classes=3):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.GELU(),
            nn.Dropout(0.30),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(256, num_classes)
        )

    def forward(self, fused):
        return self.classifier(fused)


## Cell 12 — Full PoliticalBiasModel (REDESIGN: token-level pipeline)

> **REDESIGN summary**:
> - All inputs are now token sequences `[B, L, 768]` instead of flat `[B, 768]` vectors
> - `IssueTopicContextEncoder` concatenates issue+topic tokens → `[B, Li+Lt, 768]`
> - `CrossAttentionEncoder` attends over full sequences → `[B, Li+Lt, 768]`
> - `AttentionPooling` (new) collapses to `[B, 768]` with learnable weights
> - `GatedFusion` + `BiasClassifier` unchanged


In [ ]:
class PoliticalBiasModel(nn.Module):
    """
    Issue-Topic Guided Contextual Cross-Attention Framework for Political Bias Detection.

    REDESIGNED pipeline (spec §7–§13):

        issue_tokens [B,16,768] ─┐
                                  ├─ IssueTopicContextEncoder ─→ context [B,32,768]
        topic_tokens [B,16,768] ─┘                                      │
                                                              ┌──────────┴──────────┐
        title_tokens [B,32, 768] ──────── CrossAttentionEncoder (Q=context, KV=title)
        body_tokens  [B,128,768] ──────── CrossAttentionEncoder (Q=context, KV=body)
                                                              │                     │
                                                   AttentionPooling       AttentionPooling
                                                              │                     │
                                                   headline_emb [B,768]  body_emb [B,768]
                                                              └─────────┬───────────┘
                                                                  GatedFusion [B,768]
                                                                        │
                                                                  BiasClassifier
                                                                        │
                                                              logits [B, 3]
    """
    def __init__(
        self,
        d_model            = 768,
        ctx_num_heads      = 4,
        ctx_ff_dim         = 1536,
        num_context_layers = 2,
        cross_num_heads    = 8,
        cross_ff_dim       = 3072,
        num_cross_layers   = 4,
        dropout            = 0.10,
        num_classes        = 3
    ):
        super().__init__()

        # Stage 1: Issue-Topic Context Encoder
        self.issue_topic_encoder = IssueTopicContextEncoder(
            d_model=d_model, num_heads=ctx_num_heads,
            ff_dim=ctx_ff_dim, num_layers=num_context_layers, dropout=dropout
        )

        # Stage 2a: Headline Cross-Attention Branch
        self.headline_encoder = CrossAttentionEncoder(
            num_layers=num_cross_layers, d_model=d_model,
            num_heads=cross_num_heads, ff_dim=cross_ff_dim, dropout=dropout
        )

        # Stage 2b: Body Cross-Attention Branch
        self.body_encoder = CrossAttentionEncoder(
            num_layers=num_cross_layers, d_model=d_model,
            num_heads=cross_num_heads, ff_dim=cross_ff_dim, dropout=dropout
        )

        # Stage 3: Learnable Attention Pooling (spec §11)
        self.headline_pool = AttentionPooling(d_model)
        self.body_pool     = AttentionPooling(d_model)

        # Stage 4: Gated Fusion
        self.fusion = GatedFusion(hidden_dim=d_model, dropout=dropout)

        # Stage 5: Classification
        self.classifier = BiasClassifier(input_dim=d_model, num_classes=num_classes)

    def forward(self, issue, topic, title, body):
        # issue  : [B, 16,  768]
        # topic  : [B, 16,  768]
        # title  : [B, 32,  768]
        # body   : [B, 128, 768]

        # Stage 1 — Issue-Topic Context → [B, 32, 768]
        issue_topic_ctx = self.issue_topic_encoder(issue, topic)

        # Stage 2 — Cross-attention: context queries headline/body tokens
        headline_seq = self.headline_encoder(issue_topic_ctx, title)  # [B, 32, 768]
        body_seq     = self.body_encoder(issue_topic_ctx, body)        # [B, 32, 768]

        # Stage 3 — Learnable pooling → [B, 768]
        headline_emb = self.headline_pool(headline_seq)  # [B, 768]
        body_emb     = self.body_pool(body_seq)          # [B, 768]

        # Stage 4 — Gated fusion → [B, 768]
        fused = self.fusion(headline_emb, body_emb)

        # Stage 5 — Classification → [B, 3]
        logits = self.classifier(fused)
        return logits


## Cell 13 — Instantiate & Verify Model


In [ ]:
model = PoliticalBiasModel().to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total Parameters     : {total_params:,}")
print(f"Trainable Parameters : {trainable_params:,}")
print()
print("Architecture Configuration:")
print(f"  Issue-Topic Encoder : 2 layers, 4 heads, ff_dim=1536")
print(f"  Headline Enc        : 4 layers, 8 heads, ff_dim=3072  [+AttentionPooling]")
print(f"  Body Enc            : 4 layers, 8 heads, ff_dim=3072  [+AttentionPooling]")
print(f"  Classifier          : 768 → 512 → 256 → 3")
print()

# Forward pass sanity check with correct token-sequence shapes
B = 4
dummy_issue  = torch.randn(B, 16,  768).to(device)
dummy_topic  = torch.randn(B, 16,  768).to(device)
dummy_title  = torch.randn(B, 32,  768).to(device)
dummy_body   = torch.randn(B, 128, 768).to(device)

with torch.no_grad():
    dummy_logits = model(dummy_issue, dummy_topic, dummy_title, dummy_body)

print(f"Dummy Input  — issue: [B,16,768]  topic: [B,16,768]  title: [B,32,768]  body: [B,128,768]")
print(f"Dummy Output — logits: {dummy_logits.shape}  ✅")
print("Model Instantiation : SUCCESS")


## Cell 14 — Training Loop

> Weighted `CrossEntropyLoss` | Differential LRs (DeBERTa `5e-6`, heads `2e-4`) |
> Macro-F1 early stopping (patience=10) | Gate collapse monitoring.


In [ ]:
# ── Class weights (weighted loss) ────────────────────────────────────────────
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1, 2]),
    y=labels[train_idx]
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
print(f"Class Weights — Left: {class_weights[0]:.3f} | Center: {class_weights[1]:.3f} | Right: {class_weights[2]:.3f}")

# ── Differential learning rates ───────────────────────────────────────────────
DEBERTA_LR   = 5e-6
HEADS_LR     = 2e-4
WEIGHT_DECAY = 1e-2

deberta_params = extractor.get_trainable_params()
head_params    = list(model.parameters())

optimizer = AdamW(
    [
        {"params": deberta_params, "lr": DEBERTA_LR},
        {"params": head_params,    "lr": HEADS_LR},
    ],
    weight_decay = WEIGHT_DECAY,
    betas        = (0.9, 0.999),
    eps          = 1e-8
)

# ── Training schedule ─────────────────────────────────────────────────────────
EPOCHS              = 50
GRAD_CLIP_NORM      = 1.0
EARLY_STOP_PATIENCE = 10

total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(0.10 * total_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps   = warmup_steps,
    num_training_steps = total_steps
)

criterion = CrossEntropyLoss(weight=class_weights_tensor)

# ── Training & Validation Loop ───────────────────────────────────────────────
best_val_macro_f1 = 0.0
patience_counter  = 0
gate_log          = []

for epoch in range(EPOCHS):

    # ── Train ────────────────────────────────────────────────────────────────
    model.train()
    extractor.model.train()
    train_loss, train_preds, train_labels_list = 0.0, [], []

    for batch in train_loader:
        issue        = batch["issue"].to(device)
        topic        = batch["topic"].to(device)
        title        = batch["title"].to(device)
        body         = batch["body"].to(device)
        labels_batch = batch["label"].to(device)

        optimizer.zero_grad()
        logits = model(issue, topic, title, body)
        loss   = criterion(logits, labels_batch)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            list(deberta_params) + list(head_params), max_norm=GRAD_CLIP_NORM
        )
        optimizer.step()
        scheduler.step()

        train_loss += loss.item()
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        train_preds.extend(preds)
        train_labels_list.extend(labels_batch.cpu().numpy())

    avg_train_loss = train_loss / len(train_loader)
    train_accuracy = accuracy_score(train_labels_list, train_preds)

    # ── Validate ─────────────────────────────────────────────────────────────
    model.eval()
    extractor.model.eval()
    val_loss, val_preds, val_labels_list = 0.0, [], []

    with torch.no_grad():
        for batch in val_loader:
            issue        = batch["issue"].to(device)
            topic        = batch["topic"].to(device)
            title        = batch["title"].to(device)
            body         = batch["body"].to(device)
            labels_batch = batch["label"].to(device)

            logits = model(issue, topic, title, body)
            loss   = criterion(logits, labels_batch)

            val_loss += loss.item()
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_labels_list.extend(labels_batch.cpu().numpy())

    avg_val_loss  = val_loss / len(val_loader)
    val_accuracy  = accuracy_score(val_labels_list, val_preds)
    val_macro_f1  = f1_score(val_labels_list, val_preds, average="macro", zero_division=0)

    # Gate diagnostics
    gate_mean = model.fusion.last_gate_mean
    gate_std  = model.fusion.last_gate_std
    gate_log.append((epoch + 1, gate_mean, gate_std))

    # ── Gradient verification (every 5 epochs) ───────────────────────────────
    if (epoch + 1) % 5 == 0:
        print("  ── Gradient norms ──")
        for name, module in [
            ("IssueTopicEncoder", model.issue_topic_encoder),
            ("HeadlineEncoder",   model.headline_encoder),
            ("BodyEncoder",       model.body_encoder),
            ("HeadlinePool",      model.headline_pool),
            ("GatedFusion",       model.fusion),
            ("Classifier",        model.classifier),
        ]:
            norms = [p.grad.norm().item() for p in module.parameters()
                     if p.grad is not None]
            avg_norm = sum(norms) / len(norms) if norms else 0.0
            flag = " ⚠️ VANISHING" if avg_norm < 1e-6 else ""
            print(f"    {name:20s} avg_grad_norm={avg_norm:.2e}{flag}")

    # ── Checkpoint ───────────────────────────────────────────────────────────
    if val_macro_f1 > best_val_macro_f1:
        best_val_macro_f1 = val_macro_f1
        patience_counter  = 0
        torch.save(model.state_dict(), "best_model.pt")
        saved = "✅ Saved"
    else:
        patience_counter += 1
        saved = f"(patience {patience_counter}/{EARLY_STOP_PATIENCE})"

    print(
        f"Epoch [{epoch+1:02d}/{EPOCHS}] | "
        f"Train Loss: {avg_train_loss:.4f} Acc: {train_accuracy:.4f} | "
        f"Val Loss: {avg_val_loss:.4f} Acc: {val_accuracy:.4f} | "
        f"Macro-F1: {val_macro_f1:.4f} | "
        f"Gate µ={gate_mean:.3f} σ={gate_std:.3f} | {saved}"
    )

    if patience_counter >= EARLY_STOP_PATIENCE:
        print(f"\n⏹ Early stopping at epoch {epoch+1}")
        break

print(f"\nBest Validation Macro-F1 : {best_val_macro_f1:.4f}")

print("\n── Gate Collapse Diagnosis (last 5 epochs) ──")
for ep, gm, gs in gate_log[-5:]:
    status = "⚠️  COLLAPSED" if gs < 0.05 else "✅ OK"
    print(f"  Epoch {ep:02d} | gate_mean={gm:.4f} | gate_std={gs:.4f} | {status}")


## Cell 15 — Test Set Evaluation


In [ ]:
model.load_state_dict(torch.load("best_model.pt", map_location=device))
model.eval()

test_preds, test_labels_list = [], []

with torch.no_grad():
    for batch in test_loader:
        issue  = batch["issue"].to(device)
        topic  = batch["topic"].to(device)
        title  = batch["title"].to(device)
        body   = batch["body"].to(device)
        labels_b = batch["label"].to(device)

        logits = model(issue, topic, title, body)
        preds  = torch.argmax(logits, dim=1).cpu().numpy()
        test_preds.extend(preds)
        test_labels_list.extend(labels_b.cpu().numpy())

test_accuracy = accuracy_score(test_labels_list, test_preds)
test_macro_f1 = f1_score(test_labels_list, test_preds, average="macro", zero_division=0)
print(f"Test Accuracy  : {test_accuracy:.4f}")
print(f"Test Macro-F1  : {test_macro_f1:.4f}\n")
print(classification_report(
    test_labels_list, test_preds,
    target_names=["Left", "Center", "Right"]
))
